# 4. Demo - Variability and Photometry

Learning goals:
- Include a variable -signal for one star
- Include a variable signal for multiple stars
- Activate the built-in on-board photometry module
- Extract the photometry from the HDF5 file

### Setup notebook

In [ ]:
# Reload code outside notebook
%load_ext autoreload
%autoreload 2

# Configure figures in notebook
%matplotlib notebook

### Imports

In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PlatoSim
import platosim.utilities as ut
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.matplotlibrc import setup_notebook
setup_notebook()

### Define variables

In [4]:
homeDir = os.getenv('PLATO_PROJECT_HOME')
workDir = os.getenv('PLATO_WORKDIR')

## Exercise 4.1 - Include variable sources  

PlatoSim allows you to include a variable signal for both a single target star or for each star within a subfield. To do this we need to generate a **variable source file** that can be read by PlatoSim. 

In following we will use `sim.createVariableSourceFile()` to create variable source file for a single target star. Like when we created a stellar catalogue using the simulation object, this function sets the new variable source file in the YAML tree and activates stellar variability. Let's create a simple simulation:

- <font color='orange'>Create a new simulation object</font>
- <font color='orange'>Create a stellar catalogue for a single star</font>
- <font color='orange'>Create an synthetic (sinusoidal) signal with an amplitude of a 1-2 in relative magnitudes</font>
- <font color='orange'>Inspect the docstring of the function `sim.createVariableSourceFile()` and create the synthetic variable signal</font>
- <font color='orange'>Set the number of exposures to 10,000 and decrease the dimensions of the subfield to 8 x 8 pixels</font>
- <font color='orange'>Run and visualise the simulation (use the slider)</font>

Imagine you instead want to inject a variable signal for each of the stars in your subfield. E.g. you could create three (arbitrary) synthetic variable signals yourself, however, in this example we assume that you already have created (or received) these three stellar signals. The target is gamma Doradus star with g-mode pulsations, the first contaminant star is a Eclipsing Binary (EB) system of the Algol type, and the second contaminant star is a Ap spectral type star which shows rotational modulations due to chemical percularities on its surface.

First let's download these three variable files from the PlatoSim FTP server (here we simply save the files to the `inputfiles/` directory of the Conda installation, but generally we recommend to store your files within your project folder):  

In [ ]:
path = f"{homeDir}/inputfiles"
ut.downloadFromFTP(filename="varsource_gdor.txt",  outputDir=path, server='plato')
ut.downloadFromFTP(filename="varsource_algol.txt", outputDir=path, server='plato')
ut.downloadFromFTP(filename="varsource_ap.txt",    outputDir=path, server='plato')

In order to tell PlatoSim that you want to include a variable signal for more than one star, you need to create a  **variable source list** using the function `sim.createVariableSourceList()`. In this way you have full control over which stars (by their ID) get a variable signal injected. In the following we assume all stars are variable:

- <font color='orange'>Create a new simulation object</font>
- <font color='orange'>Create a stellar catalogue containing 3 stars (e.g. with indices `[0, 1, 2]`)</font>
- <font color='orange'>Inspect the docstring of the function `sim.createVariableSourceList()` and include the 3 variable source lists (as a list)</font>
- <font color='orange'>Set the number of exposures to 10,000 and decrease the dimensions of the subfield to 8 x 8 pixels</font>
- <font color='orange'>Run and visualise the simulation (use the slider)</font>

## Exercise 4.2 - Built-in photometry module

It's time to have a look at how you can extract and visualise the light curves you produce by PlatoSim's build-in photometry module. The algorithm implemented into PlatoSim is a optimal binary mask aperture routine which also will be used for the photometric flux extraction of the so-called P5 sample (i.e. a large subset of the PIC) on-board the spacecraft.

</br>

**Pre-processing steps**

We assume that each pixel value $f_{ij}$ is composed of a signal $s_{ij}$, a background $b_{ij}$, and noise component $\epsilon_{ij}$:  

$$ f_{ij} = s_{ij} + b_{ij} + \varepsilon_{ij}$$

The implemented pre-processing steps of PlatoSim are:
- Bias subtraction (using the bias maps)
- Open shutter smearing correction (using the smearing maps)
- Convert from [ADU] to [electrons] using the (FEE and CCD) gain
- Flatfield correction (using PRNU)
- Sky background subtraction (assumed to be known)

Generally all of these steps requires measurements which introduce noise when applying each operation. The bias and smearing maps are in fact estimated from serial prescan and the parallel overscan regions, respectively. The background is assumed to be know exactly (using the background map), but in reality it will be determined from interpolation of measurements across the full FOV. Furthermore, a few steps may be a part of the future PLATO pipeline which are not yet implemented for the image reduction, being:
- CTI correction
- BFE correction
- Jitter and Drift correction



</br>

**Photometry modules**

The Optimal Aperture Photometry (OAP) method is the currenly planned photometry technique to use for stars processed on-board the PLATO spacecraft (due to limited amount of telemetry capabilities). A full description can by found in [Marchiori et al. (2019)](https://arxiv.org/abs/1906.00892)'s paper and in essence it is optimised to provide the lowest Noise-to-Signal Ratio (NSR) needed for detecting the transit signature of exoplanets. The implementation of the OAP depends on the knowledge of the PSF (position and morphology). By default PlatoSim uses inverted PSFs to determine the optimal aperture mask for each requested mask update.

We note that this method not necessarily provides the optimal description for stellar pulsation and variability (seen in the Fourier domain), but for which ultimately will be a trade-off between including as much of the stellar signal while still trying to avoid stellar contamination.

In order to compute the photometry on-board the spacecraft, PlatoSim's photometry module needs to be activated and a **photometry file**, with the index of the star(s) for which photmetry is requested, needs to be saved. Both can be accomplished using the funciton `sim.createPhotometryTargetFile()`. Just like creating a star catalogue or a variable source file, this function automatically activates the photometry module and sets the newly created file in the YAML tree. Let's try to run a simulation with the photometry extraction included:

- <font color='orange'>Create a new simulation object</font>
- <font color='orange'>Create a new stellar catalogue containing 5 stars</font>
- <font color='orange'>Inspect the docstring of the function `sim.createPhotometryTargetFile()` and create the photometry file </font>
- <font color='orange'>Turn off all output and then activate `WritePixelMaps` and `WriteStarPositions`</font>
- <font color='orange'>Simulate an imagette time series with a duration of 3 days</font>

Let's look at how we extract the photometric data. Like before to extract information from the HDF5 output file we need to use either the simulation object created with `simulation.run()` or using the class `SimFile`. We suggest to use the latter since the output file are fast fetch and you avoid using a simulation object that may have undergone some (unexpected changes). Compared to the other utilities of `SimFile`, functions extracting photometric information can also return [Pandas Data Frames](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) since these are much more convenient to work with for post-processing. Simple add the argument `df=True` to each function in the following exercise:

- <font color='orange'>Use `SimFile` to create a HDF5 output file object</font>
- <font color='orange'>Inspect the docstring of the function `f.getMaskUpdateEvents()` and extract its output</font>
- <font color='orange'>Inspect the docstring of the function `f.getApertureMask()` and extract its output</font>
- <font color='orange'>Inspect the docstring of the function `f.getCosmicsWithinApertureMask()` and extract its output</font>
- <font color='orange'>Inspect the docstring of the function `f.getTime()` and extract its output</font>
- <font color='orange'>Inspect the docstring of the function `f.getFlux()` and extract its output</font>
- <font color='orange'>Inspect the docstring of the function `f.getLightCurve()` and extract its output</font>
- <font color='orange'>Inspect the docstring of the function `f.plotLightCurve()` and plot the photometry of star ID #1</font>

## Exercise 4.3 - End-to-end simulation

It's time to combine all the knowledge gained so far and run an end-to-end simulation of a small subfield for a single camera and a full mission quarter (~91 days):  
- <font color='orange'>Copy your code from Exercie 4.1 and 4.2 (no need to do this again)</font>
- <font color='orange'>Adjust your code to only include 3 stars each with their own variable signal</font>
- <font color='orange'>Save only the pixel imagettes, star positions, and the photometry to the HDF5 file</font>
- <font color='orange'>Run your simulation (this may take tens of minutes - Jupyter notebooks are generally slow)</font>
- <font color='orange'>If time permits (perhaps after the workshop), plot the light curve of your target star</font>

<font color='orange'>**Extra exercise:**</font> 

- <font color='orange'>Similar to the end-to-end simulation above, setup a new simulation where you inject a planetary transit of a hot-Jupiter sized planet. You can download this model from our PlatoSim FTP server using the filename `varsouce_transit_hotjupiter.txt`. Try a few stellar configurations to inspect the impact that another variable source (just select one of the three from the previous exercise) has on the transit depth.</font>

**In the final tutorial we show you how to use PlatoSim's toolkit called PLATOnium to efficiently run multi-camera and multi-quarter simulations. We will do this alltogether and using the command line again.**